![EPFL Center for Imaging logo](https://imaging.epfl.ch/resources/logo-for-gitlab.svg)
# Fine-Tuning Machine Learning Models with the `transformers` Library

**Duration**: 60 min.

---
[Transformers](https://huggingface.co/docs/transformers/en/index) is a modern Python library designed to facilitate access to state-of-the-art machine learning models for vision, text, audio, and multimodal tasks. The library supports >400 model architectures based on PyTorch and enables to download and fine-tune >2M model checkpoints from the [HuggingFace Hub](https://huggingface.co/models).

This workshop will walk you through the steps of selecting a pre-trained model checkpoint, loading a training dataset, and fine-tuning a model for a specific application using `transformers`. We will focus on image classification as an example, but the concepts apply just as well to other tasks, such as image segmentation, object detection, and more.

### Installation

If you haven't done it already, install the required packages (uncomment and run this line):

In [ ]:
# !pip install transformers datasets accelerate gradio timm torch torchvision scikit-learn numpy matplotlib ipykernel ipywidgets

## Models

`Transformers` allows us to download and work with pre-trained model checkpoints from the [HuggingFace Hub](https://huggingface.co/models) platform. 

### Browsing models

On the *Hub*, models are hosted in `git` repositories that can be identified by their `<user>/<model_name>` tag known as the **model ID**. For example, the model ID [google/vit-base-patch16-224](https://huggingface.co/google/vit-base-patch16-224) hosts a Vision Transformer (ViT) model pre-trained the ImageNet-21k dataset for image classification. If you navigate to this model's page on the Hub, you will see a *Model card* providing information about the model, *Files and versions* tracked by Git, a link to the related [paper](https://huggingface.co/papers/2010.11929) and other pieces of information.

### Running a model

The easiest way to test a pretrained model is to download and run it via the `pipeline` function. We need to specify a `task` (*image-classification*, *image-segmentation*, *depth-estimation*, etc.), and a `model`, which can be a model ID from the Hub, or a path to a local model.

First, let's open an example image:

In [ ]:
from transformers.image_utils import load_image

url = "https://raw.githubusercontent.com/scikit-image/scikit-image/refs/heads/main/src/skimage/data/motorcycle_left.png"

image = load_image(url)

image

Now, let's run the pretrained image classifier (`google/vit-base-patch16-224`) on this image:

In [ ]:
from transformers import pipeline

In [ ]:
classifier = pipeline(
    task="image-classification",  # Specify the task
    model="google/vit-base-patch16-224",  # Specify the model
)

results = classifier(image, top_k=3)  # Run the pipeline

results

### Tasks

Many vision [tasks](https://huggingface.co/tasks) are available to be run with `pipeline`.

For example, to run a pre-trained model for **image segmentation**, specify `image-segmentation` as the task, and use a compatible model.

In [ ]:
# Source: www.lausanne-tourisme.ch
url = "https://static.lausanne-tourisme.ch/image/upload/f_auto,w_1200/ogvjxofbs52dgy3sdlu7"

image = load_image(url)

image

In [ ]:
segmenter = pipeline(
    task="image-segmentation", 
    model="nvidia/segformer-b1-finetuned-cityscapes-1024-1024",  # SegFormer model pre-trained on the Cityscape dataset
)

results = segmenter(image)

Let's plot our segmentation results:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

def plot_segmentation_result(results, image):
    """Plot the original image with overlaid segmentation masks for each class."""
    h, w = np.array(results[0]["mask"]).shape
    semantic_map = np.zeros((h, w), dtype=np.int32)

    labels = ["background"]
    for idx, r in enumerate(results, start=1):
        mask = np.array(r["mask"]).astype(bool)
        semantic_map[mask] = idx
        labels.append(r["label"])

    colors = [
        (0, 0, 0, 0),
        "#e41a1c",
        "#377eb8",
        "#4daf4a",
        "#984ea3",
        "#ff7f00",
        "#ffff33",
        "#a65628",
        "#f781bf",
        "#999999",
    ]
    cmap = ListedColormap(colors[:len(labels)])

    fig, ax0 = plt.subplots(figsize=(8, 8))
    ax0.imshow(image)
    ax0.imshow(semantic_map, cmap=cmap, interpolation="none", alpha=0.45, vmin=0, vmax=len(labels)-1)
    ax0.set_axis_off()
    legend_handles = [
        Patch(facecolor=colors[i], edgecolor="black", label=labels[i])
        for i in range(1, len(labels))
    ]
    ax0.legend(
        handles=legend_handles,
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        borderaxespad=0.0,
        title="Classes"
    )
    plt.tight_layout()
    plt.show()
    
plot_segmentation_result(results, image)

### Exercise

- Take a moment to browse and familiarize yourself with the [Models](https://huggingface.co/models) hub.
- (Optional) Can you run the [Depth-Anything](https://huggingface.co/depth-anything/Depth-Anything-V2-Base-hf) model to estimate a depth map on an image?

In [ ]:
# Your code here


## Datasets

Next, we will work on **fine-tuning** a model for a specific application. To do this, we first need to load a **dataset** of images, which we will use to train and evaluate our model.

The [`datasets`](https://huggingface.co/docs/datasets/index) library provides a function `load_dataset` that can be used to download datasets from the [HuggingFace Hub](https://huggingface.co/datasets) or load datasets from a local directory.

<!-- Datasets loaded in this way are backed by the [Apache Arrow](https://huggingface.co/docs/datasets/en/about_arrow) format, which enables efficient processing, even for large datasets. -->

In [ ]:
from datasets import load_dataset

HuggingFace datasets are hosted in Git repositories that can be accessed via a `<user>/<dataset_name>` **dataset ID**.

For example, to download the [Beans](https://huggingface.co/datasets/AI-Lab-Makerere/beans) dataset (360 Mb), use:

In [ ]:
dataset = load_dataset("AI-Lab-Makerere/beans")

dataset

The `Beans` dataset is a public dataset used to benchmark image classifiers trained to distinguish healthy from diseased bean leaves.

### Inspecting the dataset

Note that the `load_dataset` function returned a `DatasetDict` object. The first set of keys in this object corresponds to the **split**; typically `train`, `validation` or `test`.

In [ ]:
dataset["train"]

Each split is a separate `Dataset` object that we can index over to access the images (in PIL format) as well as the corresponding class labels:

In [ ]:
n_rows = dataset['train'].num_rows
labels = dataset['train'].features["labels"].names
id2label = {i: label for i, label in enumerate(labels)}

# Show 3 randomly selected images and their labels
fig, axes = plt.subplots(ncols=3, figsize=(15, 5))
for i, k in enumerate(np.random.choice(n_rows, 3).tolist()):
    image = dataset["train"][k]['image']
    label_idx = dataset["train"][k]['labels']
    label = id2label[label_idx]
    ax = axes.ravel()[i]
    ax.imshow(image)
    ax.set_title(f"{label}")
    ax.set_axis_off()
plt.show()

### Loading a dataset from a local folder of images

`datasets` also allow us to load a training dataset from a local directory.

For image classification, we simply need to organize the local directory according to the structure *dataset > split > classes > images*. For example:

```
ibean
|---- train
    |---- healthy
          |---- *.png
    |---- diseased
          |---- *.png
|---- validation
    |---- healthy
          |---- *.png
    |---- diseased
          |---- *.png
|---- test
|---- ...
```

We can then load the dataset using the `load_dataset` function by specifying the folder path as input.

If you have downloaded the workshop material, you should be able to load a training dataset from a local path. Here is a summary of the example datasets in the workshop material:

| Path | Dataset | Classes | License | Source |
| ----- | ----- | ----- | ----- | ----- |
| [ibean](./datasets/ibean/) | Beans | Diseased  / Healthy leaves | MIT | [↗️ link](https://huggingface.co/datasets/AI-Lab-Makerere/beans) |
| [coral](./datasets/coral/) | Corals | Bleached / Healthy corals | ODBL | [↗️ link](https://huggingface.co/datasets/aneeshd27/Corals-Classification) |
| [carpet](./datasets/carpet/) | MVTec Carpet | Defective / Normal | CC BY-SA | [↗️ link](https://www.kaggle.com/datasets/ipythonx/mvtec-ad/data) |
| [cable](./datasets/cable/) | MVTec Cable | Defective / Normal | CC BY-SA | [↗️ link](https://www.kaggle.com/datasets/ipythonx/mvtec-ad/data) |
| [leather](./datasets/leather/) | MVTec Leather | Defective / Normal | CC BY-SA | [↗️ link](https://www.kaggle.com/datasets/ipythonx/mvtec-ad/data) |
| [turbine](./datasets/turbine/) | Wind turbines | Satellite images with / without wind turbines | CC BY 4.0 | [↗️ link](https://zenodo.org/records/7385227) |



For example, to load the `Beans` dataset from a local path, use:

In [ ]:
dataset = load_dataset(path="./datasets/ibean")  # Change the path if needed

dataset

### Label mappings

Finally, it will be useful to have a correspondance between label indeces and class labels, and vice versa; so let's create these mappings now:

In [ ]:
def get_label_mappings(dataset):
    features = dataset["train"].features
    labels = features["label"].names
    label2id, id2label = dict(), dict()
    for i, label in enumerate(labels):
        label2id[label] = str(i)
        id2label[str(i)] = label
    return label2id, id2label, len(labels)
    
label2id, id2label, num_labels = get_label_mappings(dataset)

print(f"{num_labels=}")
print(id2label)
print(label2id)

## Training an image classifier

Now that we have a dataset of images that we can use for training, the next step is to choose a pre-trained model suited to our classification task.

### Choosing a pre-trained model

While we can search anywhere on the [Models](https://huggingface.co/models) hub, a good place to start is the [Pytorch Image Models](https://huggingface.co/timm/collections) collection.

It's probably best to start with a lightweight model that can be train even without a GPU. Here are some suggested model checkpoints:

| Checkpoint | Model | Params (M)  | Year |
| ---------- | ----- | ----------- | -------------- |
| [timm/resnet18.a1_in1k](https://huggingface.co/timm/resnet18.a1_in1k) | ResNet18 | 11.7 | 2015 |
| [timm/resnet50.a1_in1k](https://huggingface.co/timm/resnet50.a1_in1k) | ResNet50 | 25.6 | 2015 |
| [timm/mobilenetv3_small_100.lamb_in1k](https://huggingface.co/timm/mobilenetv3_small_100.lamb_in1k) | MobileNet | 2.5 | 2019 |
| [timm/efficientnet_b0.ra_in1k](https://huggingface.co/timm/efficientnet_b0.ra_in1k) | EfficientNet | 5.3 | 2019 |
| [timm/convnext_tiny.in12k](https://huggingface.co/timm/convnext_tiny.in12k) | ConvNext | 36.9 | 2022 |
| [timm/deit3_small_patch16_224.fb_in22k_ft_in1k](https://huggingface.co/timm/deit3_small_patch16_224.fb_in22k_ft_in1k) | DeiT-III | 22.1 | 2022 |

<!-- | [timm/mobilenetv3_large_100.ra_in1k](https://huggingface.co/timm/mobilenetv3_large_100.ra_in1k) | MobileNet | 5.5 | 2019 | -->
<!-- | [timm/tinynet_e.in1k](https://huggingface.co/timm/tinynet_e.in1k) | TinyNet | 2.0 | 2020 | -->

If you have a GPU for training, you could consider a heavier model, for example:

| Checkpoint | Model | Params (M)  | Year |
| ---------- | ----- | ----------- | -------------- |
| [timm/vit_so150m2_patch16_reg1_gap_448.sbb_e200_in12k_ft_in1k](https://huggingface.co/timm/vit_so150m2_patch16_reg1_gap_448.sbb_e200_in12k_ft_in1k) | ViT | 136.5 | 2020 |
<!-- | [apple/mobilevit-small](https://huggingface.co/apple/mobilevit-small) | MobileViT | 5.6 | 2022 | -->

In [ ]:
model_id = "timm/mobilenetv3_small_100.lamb_in1k"

<!-- ## Do we actually need to fine-tune the model? -->
### Extracting features

Classification models are typically pretrained on the [ImageNet](https://huggingface.co/datasets/ILSVRC/imagenet-1k) dataset, which contains thousands of images. As a result, they already have learned to extract meaningful features from images. Feature extraction is the process of passing images through a pre-trained model up to its penultimate layer (the layer before the classification head), then using the resulting *feature vector* for downstream analysis. In this feature space, even unseen image classes are often sufficiently separable that a simple linear model can accurately classify them.

We can use pre-trained models to extract features via `pipeline`, specifying `image-feature-extraction` as the *task*. We set `pool=True` to retreive the last hidden state after pooling: this should be a 1D feature vector.

In [ ]:
extractor = pipeline(task="image-feature-extraction", model=model_id, pool=True)

Let's run our feature extractor on the first image in the training set:

In [ ]:
example_image = dataset["train"][0]['image']

features = extractor(example_image)
features = np.array(features)

print("Features shape: ", features.shape)

Now, let's run it on all our training images (and squeeze the outputs):

In [ ]:
inputs_train = dataset["train"][:]['image']
labs_train = dataset["train"][:]['label']

embeddings_train = extractor(inputs_train , batch=32)
embeddings_train  = np.squeeze(np.array(embeddings_train))

print(f"Features shape: ", embeddings_train.shape)

If we project the feature vectors using a dimensionality reduction technique like [t-SNE](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html), we can (sometimes) already notice a separation between the different classes.

In [ ]:
import matplotlib.patches as mpatches
import matplotlib as mpl

from sklearn.manifold import TSNE

projected = TSNE(n_components=2).fit_transform(embeddings_train)

# Plot the t-SNE projection:
fig, ax = plt.subplots(figsize=(8, 8))
cmap = plt.cm.jet
norm = mpl.colors.Normalize(vmin=0, vmax=num_labels - 1)
ax.scatter(projected[:, 0], projected[:, 1], c=labs_train, cmap=cmap, norm=norm)
legend_labels = [id2label[str(i)] for i in range(num_labels)]
handles = [mpatches.Patch(color=cmap(norm(i)), label=legend_labels[i]) for i in range(num_labels)]
ax.set_xlabel("t-SNE (0)")
ax.set_ylabel("t-SNE (1)")
plt.title(f"Embeddings projected with t-SNE (shape: {projected.shape})")
plt.xticks([])
plt.yticks([])
plt.legend(handles=handles, title="Labels", loc='upper right', bbox_to_anchor=(1.3, 1))
plt.show()

### Fitting a linear classifier

To obtain a baseline classification performance, we will fit a simple linear classifier (a [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) model) to the features extracted by our pre-trained model. Note that this does *not* update the weights of the neural network: it simply fits the linear classifier to the extracted training set features.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def fit_linear_model(x_train, y_train) -> Pipeline:
    """Fit the linear model as a Scikit-learn Pipeline (incl. a standard scaler)"""
    pipe = Pipeline([
        ("scaler", StandardScaler()),  # Rescaling input features is usually a good practice
        ("model", LogisticRegression()),  # We could tweak a few parameters of the model if we wanted (like the `C` parameter which controls regularization)
    ])
    
    pipe.fit(X=x_train, y=y_train)
    
    return pipe

# Fit on the training set:
pipe = fit_linear_model(x_train=embeddings_train, y_train=labs_train)

We can evaluate our pipeline on the validation (or test) set and plot the confusion matrix:

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def apply_linear_model(model: Pipeline, x_val, y_val):
    acc = model.score(X=x_val, y=y_val)
    y_pred = model.predict(x_val)
    cm = confusion_matrix(y_val, y_pred, labels=model.classes_)
    label_names = [id2label[str(c)] for c in model.classes_]
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    
    fig_, ax = plt.subplots(figsize=(8, 8))
    disp.plot(ax=ax, colorbar=False, cmap="inferno", xticks_rotation=30)
    ax.set_title(f"Confusion matrix\nModel accuracy: {acc:.2f}")
    plt.show()

# Evaluate on the validation set (extract features => run through fitted linear model):
inputs_val = dataset["validation"][:]['image']
labs_val = dataset["validation"][:]['label']

embeddings_val = extractor(inputs_val, batch=32)
embeddings_val = np.squeeze(np.array(embeddings_val))

apply_linear_model(pipe, embeddings_val, labs_val)

It looks like, without even having fine-tunined the pre-trained model, we already have a system capable of classifying images remarkably well.

## Fine-tuning the model

To further improve the baseline performance, we will attempt to **fine-tune** the pre-trained model. To do this, we will need a way to convert images into suitable model inputs, a way to instantiate our pre-trained model, and a way to control model training.

### Converting images to model inputs

Pre-trained models do not take raw images as input, but instead expect Pytorch `Tensors`. Moreover, models typically require a fixed **input image size** (e.g. 224 x 224), with a certain intensity **normalization** being applied to the colour channels.

The `transformers` library provides us with a class `AutoImageProcessor` which can be used to automatically retreive and apply the pre-processing steps required for a particular model ([docs](https://huggingface.co/docs/transformers/en/image_processors)).

Let's instantiate an image processor for our model checkpoint:

In [ ]:
from transformers import AutoImageProcessor

image_processor = AutoImageProcessor.from_pretrained(model_id)

We can apply the image processor and compare the model inputs (resized and normalized) with the original image:

In [ ]:
image = dataset["train"][0]['image']  # First training image

inputs = image_processor(image, return_tensors="pt")

# Transpose the inputs for display as RGB (but in reality channels are first)
inputs = np.transpose(inputs['pixel_values'][0], axes=[1, 2, 0]).numpy()

fix, axes = plt.subplots(ncols=2, figsize=(10, 5))
axes[0].imshow(image)
axes[0].set_axis_off()
axes[0].set_title(f"Original image (shape: {np.array(image).shape})")
axes[1].imshow(inputs)
axes[1].set_axis_off()
axes[1].set_title(f"Model inputs (shape: {inputs.shape})")
plt.show()

### Data augmentation

We will implement a few data augmentations to help our model generalize to unseen images.

We define the transformations using the `Compose` class from `torchvision`, then we create a function to preprocess a batch of training images, and another to preprocess validation images. Preprocessing consists first of applying the augmentations (to the training batches), then converting the images into model inputs using our `image_processor`. We call `.with_transform` on the training and validation splits of the dataset so that the transformations are applied on the fly when new elements are loaded.

In [ ]:
from torchvision.transforms import Compose, RandomHorizontalFlip, RandomVerticalFlip

train_transforms = Compose([ 
    RandomHorizontalFlip(),
    RandomVerticalFlip(),
])

def preprocess_train(batch):
    images_train_aug = [train_transforms(image.convert("RGB")) for image in batch["image"]]
    inputs = image_processor(images_train_aug, return_tensors="pt")
    batch["pixel_values"] = inputs["pixel_values"]
    return batch

def preprocess_val(batch):
    images_val = [image.convert("RGB") for image in batch["image"]]
    inputs = image_processor(images_val, return_tensors="pt")
    batch["pixel_values"] = inputs["pixel_values"]
    return batch

train_ds = dataset["train"].with_transform(preprocess_train)
val_ds = dataset["validation"].with_transform(preprocess_val)

print(dataset['train'][0].keys())  # The original training set keys (images + labels)
print(train_ds[0].keys())  # Now with a 'pixel_values' column (the preprocessed images)

### Loading the pretrained model

Next, we need to load our pretrained model. We will use the `AutoModelForImageClassification` class and specify the model ID as input:

In [ ]:
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(
    model_id,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

Note that, by default, all of the model parameters are trainable. The cell below prints the number of trainable parameters of the model:

In [ ]:
def print_model_parameters(model):
    """Print a summary of trainable/frozen parameters.
    Adapted from: https://github.com/VizuaraAI/Transformers-for-vision-BOOK
    """
    trainable_params = 0
    frozen_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        num_params = param.numel()
        all_param += num_params
        if param.requires_grad:
            trainable_params += num_params
        else:
            frozen_params += num_params

    percent_trainable = 100 * trainable_params / all_param
    
    print(f"Trainable parameters:\t {trainable_params:,}")
    print(f"Frozen parameters:\t {frozen_params:,}")
    print("-"*35)
    print(f"All parameters:\t\t {all_param:,}")
    print(f"Trainable fraction (%):\t {percent_trainable:.2f}%")

print_model_parameters(model)

If we wanted, we could freeze the model weights and train only the last layer:

In [ ]:
# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the last layer (weights + bias)
for name, param in model.named_parameters():
    if "classifier" in name:
        param.requires_grad = True

print_model_parameters(model)

However, for now, we will leave all the model parameters trainable. We encourage you to revisit the code later and experiment training with frozen weights to see how this affects the results.

In [ ]:
for param in model.parameters():
    param.requires_grad = True

print_model_parameters(model)

### Last steps before training

Before we can start training the model, there are still a few things we need to set up.

To generate batches, we define a simple `collate_fn` which groups items into a batch:

In [ ]:
import torch

def collate_fn(batch):
    return {
        "pixel_values": torch.stack([item["pixel_values"] for item in batch]),
        "labels": torch.tensor([item["label"] for item in batch]),
    }

To evaluate our model during training, we set up a function to compute classification accuracy:

In [ ]:
from sklearn.metrics import accuracy_score

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

We specify a path to a directory where to save the fine-tuned model:

In [ ]:
finetuned_model_path = f"models/{model_id.split('/')[1]}"  # Feel free to change this to any path you prefer

finetuned_model_path

Lastly, we configure the training parameters using the `TrainingArguments` class, and we define a `Trainer`:

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=finetuned_model_path,  # Where to save the model
    
    # 👇 How to report evaluation metrics
    eval_strategy="epoch",  # When to evaluate. Possible values: [`steps`, `epoch`, `no`].
    save_strategy="best",  # When to save checkpoints. Possible values: [`steps`, `epoch`, `no`, `best`]
    metric_for_best_model="accuracy",  # Highest accuracy on the validation set = best (and saved) model checkpoints
    
    # 👇 Training parameters
    num_train_epochs=3,  # Number of training epochs
    learning_rate=5e-5,  # Learning rate
    per_device_train_batch_size=16,  # Batch size (training)
    per_device_eval_batch_size=16,  # Batch size (evaluation)
    
    # 👇 A few more details
    logging_steps=10,  # How to report the training loss during training
    load_best_model_at_end=True,  # Set to True so that we can evaluate the model directly after this.
    remove_unused_columns=False,  # Keep this, so we don't drop the `image` column
    
    # 👇 To force the model to be trained on CPU (e.g. in case of OOM errors)
    # use_cpu=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

print(f"Ready to train. Device: {trainer.args.device}")

### Training the model

Let's start a training run. As training progresses, we should notice a decrease in the loss while the accuracy should go up ☕.

In [ ]:
trainer.train()

### Saving the model

Once training is complete, we can save the model and the image processor to the output directory.

In [ ]:
trainer.save_model(finetuned_model_path)
image_processor.save_pretrained(finetuned_model_path)

### Model evaluation

We can now evaluate our fine-tuned model on the test dataset.

In [ ]:
x_test = dataset['test']['image']
y_test = dataset['test']['label']


To run our model, we can use the `pipeline` function, this time specifying the path to our saved model, instead of a model ID.

In [ ]:
classifier = pipeline("image-classification", finetuned_model_path)

results = classifier(list(x_test))

results[:3]  # First 3 results

Let's plot a confusion matrix to illustrate these results:

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def evaluate_finetuned_model(results, y_test):
    # Results are returned as label names, so we convert them back to indeces
    y_pred = []
    for r in results:
        imax = np.argmax([i['score'] for i in r])
        pred = r[imax]['label']
        y_pred.append(pred)
    y_pred = [int(label2id[p]) for p in y_pred]
    
    # Compute the accuracy
    acc = accuracy_score(y_pred, y_test)
    
    # Plot the confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(id2label.values()))
    
    fig_, ax = plt.subplots(figsize=(8, 8))
    disp.plot(ax=ax, colorbar=False, cmap="inferno", xticks_rotation=30)
    ax.set_title(f"Confusion matrix (test set)\nModel accuracy: {acc:.2f}")
    plt.show()

evaluate_finetuned_model(results, y_test)

It looks like we have successfully fine-tuned our model. Congratulations! 🎉

### Creating a small web app

Finally, we can create a simple web application for our image classificater using [Gradio](https://www.gradio.app/) ([docs](https://huggingface.co/docs/transformers/en/pipeline_gradio)).

Run the code below to start the web app and test your model locally in a web browser, at the URL: http://127.0.0.1:7861.

In [ ]:
import gradio as gr

classifier = pipeline("image-classification", model=finetuned_model_path)

def predict(image):
    return {x["label"]: x["score"] for x in classifier(image)}

gr.Interface(fn=predict, inputs=gr.Image(type="filepath"), outputs=gr.Label()).launch(inbrowser=True, inline=False)

## Conclusion

In this session, we have seen how the `transformers` library can help us access pre-trained models for different vision tasks, and how to fine-tune a model for image classification.

If you would like to learn more about the topics covered, we recommend that you check out:

<!-- - Use the `transformers` library to easily access and fine-tune pre-trained models for different vision tasks -->
<!-- - Use the `datasets` library to interact with datasets for training and evaluating models -->
<!-- - For image classification, a running feature extraction and a linear model can already work quite well! -->
<!-- - The `transformers` library provides a powerful workflow to fine-tune models for specific tasks -->

<!-- ### Next steps

If you would like to learn more about the topics covered, we recommend that you check out: -->

- [Transformers](https://huggingface.co/docs/transformers/en/index) - Official documentation.
- [Image segmentation](https://huggingface.co/docs/transformers/en/tasks/semantic_segmentation) - Image segmentation with the `transformers` library.
- [Customizing models](https://huggingface.co/docs/transformers/en/custom_models) - Learn how to customize model architectures.
- [Models timeline](https://huggingface.co/docs/transformers/en/models_timeline) - To keep an eye on the latest models.
- [`evaluate`](https://huggingface.co/docs/evaluate) - Evaluate and compare performance of machine learning models.
- [`trackio`](https://huggingface.co/docs/trackio) - Monitor training runs in a web dashboard (similar to `wandb` or `tensorboard`).
<!-- - [Advanced Explainable AI for computer vision](https://jacobgil.github.io/pytorch-gradcam-book/HuggingFace.html) - Using GradCam to inspect class activation maps, and more. -->